In [ ]:
import matplotlib.pyplot as plt


subscr_no_active = (
    subscr_status[subscr_status['applied'] == 1]
    [['contact_id']]
    .drop_duplicates()
)

mobapp_clicks = (
    mobapp_act[mobapp_act['metric_id'] == 1]
    [['magnit_id']]
    .drop_duplicates()
)

mobapp_clicks['has_click'] = 1

result = (
    client_cohorts
    .merge(
        subscr_no_active,
        left_on='client_id',
        right_on='contact_id',
        how='inner'
    )
    .merge(
        mobapp_clicks,
        on='magnit_id',
        how='left'
    )
)

result['has_click'] = result['has_click'].fillna(0)

cohort_clicks = (
    result
    .groupby('campaigns_cnt', as_index=False)
    .agg(
        client_cnt=('client_id', 'nunique'),
        click_cnt=('has_click', 'sum')
    )
)

cohort_clicks['click_pct'] = (
    cohort_clicks['click_cnt']
    / cohort_clicks['client_cnt']
    * 100
).round(2)

cohort_clicks

In [ ]:
plt.figure(figsize=(10, 5))

bars = plt.bar(
    cohort_clicks['campaigns_cnt'].astype(str),
    cohort_clicks['click_pct']
)

for bar, pct in zip(bars, cohort_clicks['click_pct']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{pct:.1f}%',
        ha='center'
    )

plt.title('Доля клиентов с просмотрами/кликами оффера без активной подписки')
plt.xlabel('Количество кампаний')
plt.ylabel('Доля клиентов, %')
plt.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()